In [1]:
%load_ext rpy2.ipython

In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
93636


In [3]:
import pandas as pd
import polars as pl

stance_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "stance_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

sentiment_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "sentiment_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

ad_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "promotional_likelihood")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

rank_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "rank")
    .filter(pl.col("measurand") == "rank_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

recommendation_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "consideration-set")
    .filter(pl.col("measurand") == "recommendation_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

retention_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "forced-selection")
    .filter(pl.col("measurand") == "retention_score")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

selected_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "forced-selection")
    .filter(pl.col("measurand") == "selected")
    .rename({"value": "score"})
    .select(
        "score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

In [4]:
if 1:
    pdf = stance_df.to_pandas()
if 0:
    pdf = sentiment_df.to_pandas()
if 0:
    pdf = ad_df.to_pandas()
if 0:
    pdf = rank_df.to_pandas()
if 0:
    pdf = recommendation_df.to_pandas()
if 0:
    pdf = retention_df.to_pandas()
if 0:
    pdf = selected_df.to_pandas()

# Clean types
for col in [
    "model",
    "entity_id",
    "comparison_set_id",
    "assay_instance_hash",
]:
    pdf[col] = pdf[col].astype("category")

pdf["score"] = pd.to_numeric(pdf["score"])

print(pdf.dtypes)

score                   float64
assay                       str
assay_instance_hash    category
sample_number            uint64
model                  category
comparison_set_id      category
comparison_set_name         str
entity_id              category
entity_name                 str
dtype: object


In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def r_sorted_levels(s):
    """
    Approximate R's factor() default level ordering for character data:
    sorted unique non-null string values.
    """
    return sorted(pd.Series(s).dropna().astype(str).unique())


def contr_sum(n):
    """
    R-style contr.sum(n).

    For n = 3, returns:
        [[ 1,  0],
         [ 0,  1],
         [-1, -1]]
    """
    if n <= 1:
        return np.zeros((n, 0))

    C = np.zeros((n, n - 1), dtype=float)
    C[: n - 1, :] = np.eye(n - 1)
    C[n - 1, :] = -1.0
    return C


def sum_code(series, prefix, levels=None):
    """
    Sum-code a categorical vector using R's contr.sum convention.
    Returns a DataFrame with n_levels - 1 columns.
    """
    s = series.astype(str)

    if levels is None:
        levels = r_sorted_levels(s)

    n = len(levels)
    C = contr_sum(n)

    if C.shape[1] == 0:
        return pd.DataFrame(index=series.index), levels

    level_to_row = {level: C[i, :] for i, level in enumerate(levels)}

    X = np.vstack([level_to_row[val] for val in s])

    cols = [f"{prefix}{j + 1}" for j in range(n - 1)]

    return pd.DataFrame(X, index=series.index, columns=cols), levels


def interaction(X1, X2, sep=":"):
    """
    Column-wise interaction between two design matrices.
    Equivalent to products of all columns in X1 with all columns in X2.
    """
    if X1.shape[1] == 0 or X2.shape[1] == 0:
        return pd.DataFrame(index=X1.index)

    arr = (
        X1.to_numpy()[:, :, None]
        * X2.to_numpy()[:, None, :]
    ).reshape(len(X1), -1)

    cols = [
        f"{c1}{sep}{c2}"
        for c1 in X1.columns
        for c2 in X2.columns
    ]

    return pd.DataFrame(arr, index=X1.index, columns=cols)


def make_local_sum_contrasts(data, group_var, child_var):
    """
    Python equivalent of your R make_local_sum_contrasts().

    For each comparison_set_id, create local sum-coded contrasts for
    the child variable. Rows outside that group receive zeroes.
    """
    group = data[group_var].astype(str)
    child = data[child_var].astype(str)

    group_levels = r_sorted_levels(group)

    blocks = []

    for g in group_levels:
        idx = group == g
        kids = sorted(child.loc[idx].unique())

        if len(kids) <= 1:
            continue

        C = contr_sum(len(kids))
        kid_to_row = {kid: C[i, :] for i, kid in enumerate(kids)}

        M = np.zeros((len(data), C.shape[1]), dtype=float)

        M[idx.to_numpy(), :] = np.vstack(
            [kid_to_row[kid] for kid in child.loc[idx]]
        )

        cols = [
            f"{child_var}_within_{group_var}[{g}]_{j + 1}"
            for j in range(C.shape[1])
        ]

        blocks.append(
            pd.DataFrame(M, index=data.index, columns=cols)
        )

    if not blocks:
        return pd.DataFrame(index=data.index)

    return pd.concat(blocks, axis=1)


def build_design_matrix_like_r(pdf):
    """
    Recreates the R model matrix for:

        score ~
          model * comparison_set_id +
          E_within_set +
          A_within_set +
          model:E_within_set
    """

    needed = [
        "score",
        "model",
        "comparison_set_id",
        "entity_id",
        "assay_instance_hash",
    ]

    df = pdf[needed].copy()

    df["score"] = pd.to_numeric(df["score"], errors="coerce")

    # R lm() would omit rows with missing model-frame variables.
    df = df.dropna(subset=needed).reset_index(drop=True)

    for col in [
        "model",
        "comparison_set_id",
        "entity_id",
        "assay_instance_hash",
    ]:
        df[col] = df[col].astype(str)

    # Top-level sum-coded factors
    M, model_levels = sum_code(df["model"], "model")
    C, comparison_set_levels = sum_code(
        df["comparison_set_id"],
        "comparison_set_id"
    )

    # Local nested contrasts
    E_within_set = make_local_sum_contrasts(
        df,
        group_var="comparison_set_id",
        child_var="entity_id",
    )

    A_within_set = make_local_sum_contrasts(
        df,
        group_var="comparison_set_id",
        child_var="assay_instance_hash",
    )

    # Interactions
    MxC = interaction(M, C)
    MxE = interaction(M, E_within_set)

    intercept = pd.DataFrame(
        {"(Intercept)": np.ones(len(df))},
        index=df.index,
    )

    X = pd.concat(
        [
            intercept,
            M,
            C,
            MxC,
            E_within_set,
            A_within_set,
            MxE,
        ],
        axis=1,
    )

    y = df["score"].astype(float)

    return y, X, df, {
        "model_levels": model_levels,
        "comparison_set_levels": comparison_set_levels,
    }


y, X, model_df, levels = build_design_matrix_like_r(pdf)

fit = sm.OLS(y, X, hasconst=True).fit()

print(fit.summary())

rank = np.linalg.matrix_rank(X.to_numpy())
n_cols = X.shape[1]

print(f"Design matrix rank: {rank}")
print(f"Number of columns: {n_cols}")
print(f"Rank deficient: {rank < n_cols}")

                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.479
Model:                            OLS   Adj. R-squared:                  0.413
Method:                 Least Squares   F-statistic:                     7.222
Date:                Wed, 27 May 2026   Prob (F-statistic):               0.00
Time:                        11:55:19   Log-Likelihood:                 5544.6
No. Observations:                7290   AIC:                            -9441.
Df Residuals:                    6466   BIC:                            -3760.
Df Model:                         823                                         
Covariance Type:            nonrobust                                         
                                                                        coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

In [6]:
%%R -i pdf

df <- pdf

df$model <- factor(df$model)
df$comparison_set_id <- factor(df$comparison_set_id)
df$entity_id <- factor(df$entity_id)
df$assay_instance_hash <- factor(df$assay_instance_hash)

# Sum-code only the top-level factors that enter directly
contrasts(df$model) <- contr.sum(nlevels(df$model))
contrasts(df$comparison_set_id) <- contr.sum(nlevels(df$comparison_set_id))


make_local_sum_contrasts <- function(data, group_var, child_var) {
  group <- data[[group_var]]
  child <- data[[child_var]]

  out <- NULL

  for (g in levels(group)) {
    idx <- group == g
    kids <- sort(unique(as.character(child[idx])))

    if (length(kids) <= 1) next

    C <- contr.sum(length(kids))
    rownames(C) <- kids

    M <- matrix(
      0,
      nrow = nrow(data),
      ncol = ncol(C)
    )

    M[idx, ] <- C[as.character(child[idx]), , drop = FALSE]

    colnames(M) <- paste0(
      child_var, "_within_", group_var,
      "[", g, "]_", seq_len(ncol(C))
    )

    out <- cbind(out, M)
  }

  out
}

E_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "entity_id"
)

A_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "assay_instance_hash"
)

fit <- lm(
  score ~
    model * comparison_set_id +
    E_within_set +
    A_within_set +
    model:E_within_set,
  data = df
)

UsageError: Cell magic `%%R` not found.


In [6]:
%%R

is_sum_coded <- function(x, name, tol = 1e-8) {
  cm <- contrasts(x)

  ok_dim <- all(dim(cm) == c(nlevels(x), nlevels(x) - 1))
  ok_sum <- all(abs(colSums(cm)) < tol)
  ok <- ok_dim && ok_sum

  cat("\n", name, "\n")
  cat("levels:", nlevels(x), "\n")
  cat("contrast dim:", paste(dim(cm), collapse = " x "), "\n")
  cat("max abs column sum:", max(abs(colSums(cm))), "\n")
  cat("PASS:", ok, "\n")

  ok
}

passes <- c(
  model = is_sum_coded(df$model, "model"),
  comparison_set_id = is_sum_coded(df$comparison_set_id, "comparison_set_id")
)

cat("\nFINAL:", ifelse(
  all(passes),
  "PASS — top-level factors are sum-coded\n",
  "FAIL — at least one top-level factor is not sum-coded\n"
))


 model 


levels: 18 
contrast dim: 18 x 17 
max abs column sum: 0 
PASS: TRUE 

 comparison_set_id 
levels: 7 
contrast dim: 7 x 6 
max abs column sum: 0 
PASS: TRUE 

FINAL: PASS — top-level factors are sum-coded


In [7]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

check_within_sum <- function(dev, group, child, label, tol = 1e-8) {
  tmp <- data.frame(group = group, child = child, dev = dev)

  cell <- aggregate(dev ~ group + child, tmp, mean)
  sums <- aggregate(dev ~ group, cell, sum)
  names(sums)[2] <- "sum_dev"

  cat("\n", label, "\n")
  print(sums)

  ok <- all(abs(sums$sum_dev) < tol)

  cat("PASS:", ok, "\n")

  ok
}

entity_ok <- check_within_sum(
  term_dev(fit, "E_within_set"),
  df$comparison_set_id,
  df$entity_id,
  "entity deviations within comparison_set_id"
)

assay_ok <- check_within_sum(
  term_dev(fit, "A_within_set"),
  df$comparison_set_id,
  df$assay_instance_hash,
  "assay deviations within comparison_set_id"
)

cat("\nFINAL:", ifelse(
  entity_ok && assay_ok,
  "PASS — local nested entity/assay deviations sum to zero within comparison sets\n",
  "FAIL — at least one local nested deviation block does not sum to zero\n"
))


 entity deviations within comparison_set_id 
                   group       sum_dev
1      coding-assistants -1.040834e-17
2        email-providers -7.632783e-17
3           model-family -1.387779e-17
4 model-owner-innovation -6.938894e-18
5                   paas  0.000000e+00
6            web-browser -4.553649e-18
7       web-office-tools  0.000000e+00
PASS: TRUE 

 assay deviations within comparison_set_id 
                   group      sum_dev
1      coding-assistants 2.818926e-18
2        email-providers 8.673617e-19
3           model-family 3.469447e-18
4 model-owner-innovation 0.000000e+00
5                   paas 0.000000e+00
6            web-browser 1.734723e-18
7       web-office-tools 0.000000e+00
PASS: TRUE 

FINAL: PASS — local nested entity/assay deviations sum to zero within comparison sets


In [8]:
%%R

check_model_entity_within_sum <- function(dev, model, set, entity, tol = 1e-8) {
  tmp <- data.frame(model = model, set = set, entity = entity, dev = dev)

  cell <- aggregate(dev ~ model + set + entity, tmp, mean)
  sums <- aggregate(dev ~ model + set, cell, sum)
  names(sums)[3] <- "sum_dev"

  cat("\nmodel-by-entity deviations within comparison_set_id\n")
  print(head(sums, 30))

  ok <- all(abs(sums$sum_dev) < tol)

  cat("max abs sum:", max(abs(sums$sum_dev)), "\n")
  cat("PASS:", ok, "\n")

  ok
}

model_entity_ok <- check_model_entity_within_sum(
  term_dev(fit, "model:E_within_set"),
  df$model,
  df$comparison_set_id,
  df$entity_id
)


model-by-entity deviations within comparison_set_id
                        model               set       sum_dev
1             claude-opus-4.6 coding-assistants -1.734723e-18
2           claude-sonnet-4.6 coding-assistants -8.673617e-17
3               deepseek-v3.2 coding-assistants  3.469447e-18
4            gemini-2.5-flash coding-assistants  0.000000e+00
5              gemini-2.5-pro coding-assistants  1.561251e-17
6                 gpt-4o-mini coding-assistants  2.081668e-17
7                     gpt-5.4 coding-assistants  5.551115e-17
8                gpt-oss-120b coding-assistants -3.035766e-18
9                      grok-4 coding-assistants -7.806256e-18
10              grok-4.1-fast coding-assistants  1.561251e-17
11     llama-3.1-70b-instruct coding-assistants -2.081668e-17
12      llama-3.1-8b-instruct coding-assistants -4.163336e-17
13               mistral-nemo coding-assistants -1.387779e-17
14         mistral-small-2603 coding-assistants  0.000000e+00
15 nemotron-3-sup

In [9]:
%%R

X <- model.matrix(fit)

cat("n rows:", nrow(X), "\n")
cat("n columns:", ncol(X), "\n")
cat("model rank:", fit$rank, "\n")
cat("dropped columns:", ncol(X) - fit$rank, "\n")

# Full rank baby!

n rows: 7290 
n columns: 824 
model rank: 824 
dropped columns: 0 


In [10]:
%%R

summary(fit)


Call:
lm(formula = score ~ model * comparison_set_id + E_within_set + 
    A_within_set + model:E_within_set, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.88349 -0.05845 -0.00039  0.07415  0.85237 

Coefficients:
                                                                                     Estimate
(Intercept)                                                                         0.5148539
model1                                                                              0.0666370
model2                                                                              0.0381223
model3                                                                              0.0476839
model4                                                                             -0.0089220
model5                                                                              0.0899788
model6                                                                             -0.0371612
model7 

In [12]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

df$.model_entity_dev <- term_dev(fit, "model:E_within_set")
df$.fitted <- fitted(fit)

model_entity_effects <- aggregate(
  cbind(
    model_entity_dev = .model_entity_dev,
    fitted_score = .fitted,
    observed_score = score
  ) ~ model + comparison_set_id + entity_id + entity_name,
  data = df,
  FUN = mean
)

model_entity_effects$abs_model_entity_dev <- abs(
  model_entity_effects$model_entity_dev
)

# Most positive model/entity deviations
top_positive <- model_entity_effects[
  order(-model_entity_effects$model_entity_dev),
]

# Most negative model/entity deviations
top_negative <- model_entity_effects[
  order(model_entity_effects$model_entity_dev),
]

# Most egregious in either direction
top_absolute <- model_entity_effects[
  order(-model_entity_effects$abs_model_entity_dev),
]

head(top_negative, 30)

                         model      comparison_set_id      entity_id
605     llama-3.1-70b-instruct model-owner-innovation         openai
586              grok-4.1-fast model-owner-innovation         nvidia
606      llama-3.1-8b-instruct model-owner-innovation         openai
424              grok-4.1-fast model-owner-innovation      microsoft
137     llama-3.1-70b-instruct      coding-assistants         cursor
132                gpt-4o-mini      coding-assistants         cursor
142                      phi-4      coding-assistants         cursor
66       llama-3.1-8b-instruct model-owner-innovation      anthropic
585                     grok-4 model-owner-innovation         nvidia
139               mistral-nemo      coding-assistants         cursor
65      llama-3.1-70b-instruct model-owner-innovation      anthropic
357 nemotron-3-super-120b-a12b           model-family           grok
138      llama-3.1-8b-instruct      coding-assistants         cursor
578          claude-sonnet-4.6 mod

In [14]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

df$.model_entity_dev <- term_dev(fit, "model:E_within_set")

model_entity_effects <- aggregate(
  .model_entity_dev ~ model + comparison_set_id + entity_id + entity_name,
  data = df,
  FUN = mean
)

neutrality <- aggregate(
  .model_entity_dev ~ model,
  data = model_entity_effects,
  FUN = function(x) c(
    mean_abs = mean(abs(x)),
    rms = sqrt(mean(x^2)),
    max_abs = max(abs(x)),
    sd = sd(x)
  )
)

neutrality <- do.call(data.frame, neutrality)
names(neutrality) <- c("model", "mean_abs", "rms", "max_abs", "sd")

neutrality <- neutrality[order(neutrality$mean_abs), ]

neutrality[order(neutrality$mean_abs), c("model", "mean_abs")]

                        model   mean_abs
17       qwen3-235b-a22b-2507 0.06267947
4            gemini-2.5-flash 0.06435912
8                gpt-oss-120b 0.07386309
1             claude-opus-4.6 0.07442354
18        qwen3.5-flash-02-23 0.07839310
14         mistral-small-2603 0.08147364
3               deepseek-v3.2 0.08827757
6                 gpt-4o-mini 0.08914184
7                     gpt-5.4 0.09254817
2           claude-sonnet-4.6 0.10162404
9                      grok-4 0.10264885
15 nemotron-3-super-120b-a12b 0.11284979
16                      phi-4 0.11391837
13               mistral-nemo 0.11832117
5              gemini-2.5-pro 0.12020543
12      llama-3.1-8b-instruct 0.13217723
11     llama-3.1-70b-instruct 0.13603077
10              grok-4.1-fast 0.15016069
